# 🚀 Deploy XGBoost Model to AWS SageMaker

**What this notebook does:**
1. Uploads `model.tar.gz` to a specified S3 location (if it isn’t already there).
2. Uses Boto3 to create a SageMaker Model pointing at the official XGBoost container in `ca-central-1`.
3. Creates a SageMaker EndpointConfig (1 instance of your choosing).
4. Launches a SageMaker Endpoint and polls until it becomes `InService`.

> **Prerequisites**:
> - You have AWS credentials configured (either via `~/.aws/credentials` or environment variables).
> - You know the SageMaker execution role ARN (e.g. `arn:aws:iam::222634404112:role/SageMakerExecutionRole-ca`).
> - The IAM Role has permissions to read your S3 bucket, create SageMaker resources, and pull the XGBoost container from ECR.
> - You are running in a region that supports XGBoost (we’ll assume `ca-central-1`).


## 1) Environment Setup

Import necessary packages, set your AWS region, role ARN, bucket name, and S3 prefix.


In [8]:
# %% [code]
import os
import time
import boto3
import sagemaker
from sagemaker import get_execution_role
from sagemaker.session import Session
from sagemaker.image_uris import retrieve
from pathlib import Path
import json

# ------------------------------
# 1.1) AWS / SageMaker Configuration
# ------------------------------
REGION = "ca-central-1"   # Change if you are in another region

# Replace with your actual SageMaker execution role
# (this Role must have: s3:GetObject, s3:PutObject on your bucket,
#  plus SageMakerFullAccess or equivalent, and ECR pull permissions.
#  If you call `sagemaker.get_execution_role()`, it will return the role 
#  for the current notebook instance—but if you run outside a SageMaker notebook
#  you must hard‐code the role ARN.)
SAGEMAKER_ROLE_ARN = "arn:aws:iam::222634404112:role/SageMakerExecutionRole-ca"

# ------------------------------
# 1.2) S3 Bucket & Prefix
# ------------------------------
# (The notebook will upload to: s3://{S3_BUCKET}/{S3_PREFIX}/model.tar.gz)
S3_BUCKET = "my-aki-model-bucket"   # ◀︎ REPLACE with your actual bucket
S3_PREFIX = "aki-risk"

# Construct full S3 URI to where we will place model.tar.gz
s3_model_key = f"{S3_PREFIX}/model.tar.gz"
s3_model_uri = f"s3://{S3_BUCKET}/{s3_model_key}"

# ------------------------------
# 1.3) Local Path to Your tar.gz
# ------------------------------
LOCAL_MODEL_TAR = Path("model.tar.gz")
if not LOCAL_MODEL_TAR.exists():
    raise FileNotFoundError(f"model.tar.gz not found in current dir: {LOCAL_MODEL_TAR.resolve()}")

# ------------------------------
# 1.4) Boto3 / SageMaker Clients
# ------------------------------
boto3.setup_default_session(region_name=REGION)
sagemaker_client = boto3.client("sagemaker", region_name=REGION)
s3_client = boto3.client("s3", region_name=REGION)

# Create a SageMaker Session object (used by the high‐level SDK)
sm_session = sagemaker.Session(boto_session=boto3.Session(region_name=REGION))

print("✔️  Region:       ", REGION)
print("✔️  SageMaker Role:", SAGEMAKER_ROLE_ARN)
print("✔️  S3 Bucket:    ", S3_BUCKET)
print("✔️  Will upload to:", s3_model_uri)


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pydantic/_internal/_fields.py:198: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


[06/04/25 21:33:41] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=412788;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=10458;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py#1352\1352]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/manraj/Library/Application Support/sagemaker/config.yaml


[06/04/25 21:33:42] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=906344;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=780448;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py#1352\1352]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=145813;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=214045;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py#1352\1352]8;;\

✔️  Region:        ca-central-1
✔️  SageMaker Role: arn:aws:iam::222634404112:role/SageMakerExecutionRole-ca
✔️  S3 Bucket:     my-aki-model-bucket
✔️  Will upload to: s3://my-aki-model-bucket/aki-risk/model.tar.gz


## 2) Upload `model.tar.gz` to S3 (if not already there)

We’ll upload to: `s3://{S3_BUCKET}/{S3_PREFIX}/model.tar.gz`.  
(If the object already exists, we simply skip upload.)

In [9]:
# %% [code]
print("Checking S3 for existing artifact...")

try:
    s3_client.head_object(Bucket=S3_BUCKET, Key=s3_model_key)
    print(f"✅ Found existing S3 object: {s3_model_uri}")
except s3_client.exceptions.NoSuchKey:
    print(f"✋ Key not found; uploading local → {s3_model_uri} ...")
    s3_client.upload_file(
        Filename=str(LOCAL_MODEL_TAR),
        Bucket=S3_BUCKET,
        Key=s3_model_key
    )
    print("✅ Upload complete.")
except Exception as e:
    # If the bucket doesn’t even exist, you’ll see a 404 on bucket, or AccessDenied.
    raise


Checking S3 for existing artifact...
✅ Found existing S3 object: s3://my-aki-model-bucket/aki-risk/model.tar.gz


In [11]:
# %% [code]
# In ca-central-1, the official AWS XGBoost v1.6‐1 CPU inference image URI is:
xgb_inference_image = retrieve(
    framework="xgboost",
    region=REGION,
    version="1.5-1",
    image_scope="inference"
)

print("✔️ Using XGBoost inference image URI:")
print("  ", xgb_inference_image)


[06/04/25 21:34:12] INFO     Ignoring unnecessary instance type: None.                            ]8;id=576366;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=171445;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/image_uris.py#530\530]8;;\

✔️ Using XGBoost inference image URI:
   341280168497.dkr.ecr.ca-central-1.amazonaws.com/sagemaker-xgboost:1.5-1


In [12]:
# %% [code]
MODEL_NAME = "aki-xgb-model"   # ▲ Change if you want something else

print(f"▶ Creating SageMaker Model \"{MODEL_NAME}\"...")

create_model_response = sagemaker_client.create_model(
    ModelName=MODEL_NAME,
    PrimaryContainer={
        "Image": xgb_inference_image,
        "ModelDataUrl": s3_model_uri,
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",   # must match your inference script name
            "SAGEMAKER_REGION": REGION
        },
    },
    ExecutionRoleArn=SAGEMAKER_ROLE_ARN,
)

print("✔️  Created Model ARN:")
print(json.dumps(create_model_response["ModelArn"], indent=4))


▶ Creating SageMaker Model "aki-xgb-model"...
✔️  Created Model ARN:
"arn:aws:sagemaker:ca-central-1:222634404112:model/aki-xgb-model"


In [13]:
# %% [code]
ENDPOINT_CONFIG_NAME = "aki-xgb-endpoint-config"

print(f"▶ Creating EndpointConfig \"{ENDPOINT_CONFIG_NAME}\"...")

create_endpoint_config_response = sagemaker_client.create_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": MODEL_NAME,
            "InitialInstanceCount": 1,
            "InstanceType": "ml.m5.large",
            "InitialVariantWeight": 1.0,
        }
    ],
)

print("✔️  Created EndpointConfig ARN:")
print(json.dumps(create_endpoint_config_response["EndpointConfigArn"], indent=4))


▶ Creating EndpointConfig "aki-xgb-endpoint-config"...
✔️  Created EndpointConfig ARN:
"arn:aws:sagemaker:ca-central-1:222634404112:endpoint-config/aki-xgb-endpoint-config"


In [ ]:
# %% [code]
ENDPOINT_NAME = "aki-xgb-endpoint"

print(f"▶ Creating SageMaker Endpoint \"{ENDPOINT_NAME}\"...")

create_endpoint_response = sagemaker_client.create_endpoint(
    EndpointName=ENDPOINT_NAME,
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
)
print("✔️  CreateEndpoint API response:", json.dumps(create_endpoint_response["EndpointArn"], indent=4))

print("⏳ Waiting for endpoint to become InService… (may take ~5–10 minutes)")

while True:
    desc = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    status = desc["EndpointStatus"]
    print(f"   • Status: {status}")
    if status == "InService":
        break
    elif status in ["Failed", "RollingBack", "OutOfService"]:
        raise RuntimeError(f"Endpoint creation failed with status: {status}")
    time.sleep(30)

print(f"\n✅ Endpoint \"{ENDPOINT_NAME}\" is now InService!")


▶ Creating SageMaker Endpoint "aki-xgb-endpoint"...
✔️  CreateEndpoint API response: "arn:aws:sagemaker:ca-central-1:222634404112:endpoint/aki-xgb-endpoint"
⏳ Waiting for endpoint to become InService… (may take ~5–10 minutes)
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating
   • Status: Creating


In [ ]:
# %% [code]
import json

runtime_sm_client = boto3.client("sagemaker-runtime", region_name=REGION)

# ▶ Build a dummy payload with *all* 12 features as strings or numbers
test_payload = {
    "HCO3": 25,
    "Creatinine": 1.2,
    "Mean Arterial Pressure": 90,
    "Procalcitonin": 0.05,
    "Bilirubin": 0.7,
    "pH": 7.38,
    "Albumin": 3.5,
    "Urea": 18,
    "White Blood Cell Count": 8.0,
    "SOFA": 2,
    "APACHEII": 12,
    "Glasgow": 15
}

print("▶ Invoking endpoint with payload:", test_payload)
response = runtime_sm_client.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps(test_payload),
)

# Read and decode the response
result = response["Body"].read().decode("utf-8")
print("✔️  Raw JSON response:")
print(result)

# If your `inference.py` returns e.g. {"prediction": 0.42}, you can load it:
try:
    result_dict = json.loads(result)
    print("\n▶ Parsed output (as dict):", result_dict)
except json.JSONDecodeError:
    print("\n▶ Warning: Response was not valid JSON.")


## 3) Create a SageMaker Model

Here we tell SageMaker:
  - **Image**: `246618743249.dkr.ecr.ca-central-1.amazonaws.com/xgboost:1.6-1-cpu-py3`  
    (this is AWS’s official XGBoost inference container in `ca-central-1`)
  - **ModelDataUrl**: `s3://{S3_BUCKET}/{S3_PREFIX}/model.tar.gz`  
  - **Environment**: It should run `inference.py` inside your tarball, so we pass `SAGEMAKER_PROGRAM=inference.py`.
  - **ExecutionRoleArn**: The SageMaker role that has S3/ECR/SageMaker permissions.

In [ ]:
import boto3, json, time
from botocore.exceptions import ClientError

# ————————————————
# 1) Configuration
# ————————————————
REGION = "ca-central-1"
SAGEMAKER_ROLE_ARN = "arn:aws:iam::222634404112:role/SageMakerExecutionRole-ca"
S3_BUCKET       = "my-aki-model-bucket"
S3_PREFIX       = "aki-risk"
MODEL_NAME      = "aki-xgb-model"

boto3.setup_default_session(region_name=REGION)
sm = boto3.client("sagemaker", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)

# Full S3 URI to your already-uploaded model.tar.gz
s3_model_key = f"{S3_PREFIX}/model.tar.gz"
s3_model_uri = f"s3://{S3_BUCKET}/{s3_model_key}"

# ————————————————
# 2) Create the SageMaker Model
# ————————————————
print(f"Creating SageMaker model \"{MODEL_NAME}\"...")
create_model_response = sm.create_model(
    ModelName=MODEL_NAME,
    PrimaryContainer={
        # <-- this must be the official AWS‐managed XGBoost URI for ca-central-1:
        "Image": "683313688378.dkr.ecr.ca-central-1.amazonaws.com/xgboost:1.6-1-cpu-py3",
        "ModelDataUrl": s3_model_uri,
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_REGION":  REGION
        },
    },
    ExecutionRoleArn=SAGEMAKER_ROLE_ARN
)
print(json.dumps(create_model_response, indent=2, default=str))


In [ ]:
print(f"Creating SageMaker model \"{MODEL_NAME}\"...")
create_model_response = sm.create_model(
    ModelName=MODEL_NAME,
    PrimaryContainer={
        # <-- this must be the official AWS‐managed XGBoost URI for ca-central-1:
        "Image": "683313688378.dkr.ecr.ca-central-1.amazonaws.com/xgboost:1.6-1-cpu-py3",
        "ModelDataUrl": s3_model_uri,
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_REGION":  REGION
        },
    },
    ExecutionRoleArn=SAGEMAKER_ROLE_ARN
)
print(json.dumps(create_model_response, indent=2, default=str))

## 4) Create an Endpoint Configuration

We’ll spin up a single `ml.m5.large` instance (feel free to change instance type).  
The `ProductionVariants` block tells SageMaker:
  - `VariantName`: any name you like (e.g. `AllTraffic`)  
  - `ModelName`: must match the model you created above (`aki-xgb-model`)  
  - `InitialInstanceCount`: how many instances (we’ll use 1)  
  - `InstanceType`: e.g. `ml.m5.large`  
  - `InitialVariantWeight`: relative traffic weight, `1.0` means 100% of incoming traffic goes here.

In [ ]:
# 4.1) Name your EndpointConfig
ENDPOINT_CONFIG_NAME = "aki-xgb-endpoint-config"

print(f"Creating endpoint configuration \"{ENDPOINT_CONFIG_NAME}\"...")
create_endpoint_config_response = sagemaker_client.create_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
    ProductionVariants=[
        {
            'VariantName': 'AllTraffic',
            'ModelName': MODEL_NAME,
            'InitialInstanceCount': 1,
            'InstanceType': 'ml.m5.large',
            'InitialVariantWeight': 1.0
        }
    ]
)

print(json.dumps(create_endpoint_config_response, indent=4, default=str))


## 5) Create & Launch the Endpoint

This call returns immediately, but the endpoint will take several minutes to come up.
We’ll then poll `DescribeEndpoint` until `EndpointStatus == "InService"`.

In [ ]:
# 5.1) Name your Endpoint
ENDPOINT_NAME = "aki-xgb-endpoint"

print(f"Creating SageMaker endpoint \"{ENDPOINT_NAME}\"...")
create_endpoint_response = sagemaker_client.create_endpoint(
    EndpointName=ENDPOINT_NAME,
    EndpointConfigName=ENDPOINT_CONFIG_NAME
)
print(json.dumps(create_endpoint_response, indent=4, default=str))

# 5.2) Poll until the endpoint is InService
print("Waiting for endpoint to be InService… (this may take 5–10 minutes)")
while True:
    desc = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    status = desc['EndpointStatus']
    print(f"   Status: {status}")
    if status == 'InService':
        break
    elif status in ['Failed','RollingBack','Deleting']:
        raise RuntimeError(f"Endpoint creation failed or was deleted: {status}")
    time.sleep(30)

print(f"\n✅ Endpoint \"{ENDPOINT_NAME}\" is InService! 🎉")


## 6) (Optional) Test Your Endpoint

Now that your endpoint is live, you can invoke it with some dummy data:
```python
import boto3, json

runtime = boto3.client('sagemaker-runtime', region_name=REGION)
payload = json.dumps({
   "HCO3": 25,
   "Creatinine": 1.2,
   "Mean Arterial Pressure": 90,
   ... all 12 features ...
})
response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType='application/json',
    Body=payload
)
result = json.loads(response['Body'].read().decode())
print("Prediction result:", result)
```

Your `inference.py` should return a JSON object (e.g. `{ 'prediction': 0.42 }` for a 42% AKI risk).  

_End of notebook._